# Notebook 01 — Dataset Audit
**Project:** AI-Based Insider Threat Detection System Using Unsupervised Anomaly Detection

This notebook audits `data/raw/master_behavioral_matrix.parquet` and `data/raw/feature_matrix_v1.parquet` independently, checking schema, dimensions, memory usage, missing values, duplicates, ground-truth label availability, and data leakage risks.

In [ ]:
import json
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path('..').resolve()
audit_path = PROJECT_ROOT / 'reports' / 'metrics' / 'dataset_audit_report.json'
audit = json.loads(audit_path.read_text(encoding='utf-8'))
print('Loaded audit report from:', audit_path)

In [ ]:
for key in ['master_behavioral_matrix', 'feature_matrix_v1']:
    r = audit[key]
    print(f"=== {r['file_name']} ===")
    print(f"Disk: {r['file_size_mb']} MB | RAM: {r['memory_usage_mb']} MB | Shape: ({r['row_count']}, {r['column_count']})")
    print(f"Employees: {r['unique_employees']} | Dates: {r['date_info']['unique_dates']} ({r['date_info']['min_date']} to {r['date_info']['max_date']})")
    print(f"Duplicates: rows={r['duplicate_rows']}, emp-date={r['duplicate_employee_date_rows']}")
    print(f"Missing values: {r['missing_values']}")
    print(f"Potential label columns: {r['potential_label_columns']}")
    print()

## Key Audit Conclusions & Limitations
1. **Primary Dataset (`master_behavioral_matrix.parquet`)**: 330,452 employee-day rows across 1,000 employees and 501 dates (`2010-01-02` to `2011-05-17`). Zero missing values in numeric behavioural columns; missingness only in `role` (31,281), `department` (36,125), and `functional_unit` (31,973).
2. **Reference Dataset (`feature_matrix_v1.parquet`)**: Contains 16 additional pre-engineered columns, including `*_z_user` columns with `mean = 0.0` and zero missingness, indicating full-timeline per-user standardization (temporal leakage). Therefore, it is **not** used for model training.
3. **Ground-Truth Labels**: Neither file contains ground-truth insider threat labels. Supervised evaluation metrics cannot be computed without external CERT r4.2 `answers/` annotations.